# 4. Latih model

Tiga algoritma dilatih pada data dan pembagian yang sama: Random Forest, XGBoost, dan SVM.
Tuning tidak dilakukan berulang kali pada data uji; pemilihan model memakai data validasi.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from train_model import build_models, evaluate, load_data, split_data

X, y, groups, audit = load_data()
idx = split_data(X, y, groups)
print("Ukuran split:", {k: len(v) for k, v in idx.items()})

X_latih, y_latih = X.iloc[idx["train"]], y.iloc[idx["train"]]
X_validasi, y_validasi = X.iloc[idx["val"]], y.iloc[idx["val"]]

Ukuran split: {'train': 81483, 'val': 10443, 'test': 8946, 'train_val': 91926}


In [2]:
hasil = {}
for nama, model in build_models().items():
    if nama == "SVM":
        sampel = X_latih.sample(n=min(30000, len(X_latih)), random_state=42).index
        data_latih, label_latih = X_latih.loc[sampel], y_latih.loc[sampel]
    else:
        data_latih, label_latih = X_latih, y_latih

    model.fit(data_latih, label_latih)
    hasil[nama] = evaluate(model, X_validasi, y_validasi)
    print(nama, "F1 =", round(hasil[nama]["f1"], 4), "AUC =", round(hasil[nama]["roc_auc"], 4))

Random Forest F1 = 0.9994 AUC = 1.0


XGBoost F1 = 0.9993 AUC = 1.0


SVM F1 = 0.9988 AUC = 1.0


In [3]:
import pandas as pd

ringkasan = pd.DataFrame(hasil).T[["accuracy", "precision", "recall", "f1", "roc_auc", "false_negative_rate"]]
print(ringkasan.round(4).to_string())
print("\nModel terpilih:", ringkasan["f1"].idxmax())

               accuracy precision    recall        f1   roc_auc false_negative_rate
Random Forest   0.99933  0.998814       1.0  0.999407  0.999998                 0.0
XGBoost        0.999234  0.998983  0.999661  0.999322  0.999998            0.000339
SVM            0.998659  0.999321  0.998304  0.998812  0.999981            0.001696

Model terpilih: Random Forest
